## 10.2 MLP 案例 - 模型搭建、优化器定义、学习率调度与训练

In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. 数据预处理
transform = transforms.ToTensor() # 将图像转换为张量

# 2. 加载数据集
train_dataset = datasets.MNIST(
    root='./data', # 数据存储路径
    train=True, # 是否加载训练集
    transform=transform, # 应用数据预处理
    download=True # 如果数据集不存在，是否下载
)
test_dataset = datasets.MNIST(
    root='./data', # 数据存储路径
    train=False, # 是否加载测试集
    transform=transform, # 应用数据预处理
    download=True # 如果数据集不存在，是否下载
)

# 3. 创建数据加载器
train_loader = DataLoader(
    dataset=train_dataset, # 训练数据集
    batch_size=64, # 每个批次的样本数量
    shuffle=True # 是否打乱数据
)
test_loader = DataLoader(
    dataset=test_dataset, # 测试数据集
    batch_size=64, # 每个批次的样本数量
    shuffle=False # 是否打乱数据
)

#### 1. 这一节我们要完成什么
1️⃣ 从数据准备过渡到模型训练

在上一节中，我们已经完成了数据准备，知道了：
* 数据集使用的是 MNIST
* 每张图片大小是 28 × 28
* 输入到 MLP 前需要展平为 784 维向量
* 这是一个 10 分类问题

👉 定义模型 → 定义损失函数 → 定义优化器 → 定义学习率调度器 → 编写训练循环


##### 1.2 这一节的目标
* MLP 模型类应该怎么写
* 为什么输入层是 784，输出层是 10
* 为什么隐藏层后面可以加入 Batch Normalization
* 为什么优化器定义在模型外面
* 为什么 scheduler 也定义在模型外面
* 一次训练循环到底做了什么
* forward()、loss.backward()、optimizer.step()、scheduler.step() 之间是什么关系
* 为什么这里不用手动写 Softmax
* 为什么这里不用手动把标签转成 one-hot

#### 2. 先回顾整个训练流程

##### 2.1 神经网络训练的完整顺序
1. 准备数据
2. 定义模型
3. 定义损失函数
4. 定义优化器
5. 定义学习率调度器（可选但常用）
6. 循环训练：
   - 前向传播
   - 计算损失
   - 梯度清零
   - 反向传播
   - 参数更新
   - 调整学习率（按策略）

##### 2.2 这几个步骤分别在做什么
（1）定义模型

决定网络结构，也就是：
* 输入层有多少神经元
* 隐藏层有多少层
* 每层多少神经元
* 使用什么激活函数

（2）定义损失函数

用来衡量：

👉 模型预测得有多差

损失越小，说明预测越接近真实结果。

（3）定义优化器

优化器负责根据梯度来更新参数，例如：
* 权重 W
* 偏置 b

（4）定义学习率调度器 scheduler

scheduler 的作用是：

👉 随着训练进行，动态调整学习率

因为训练初期通常希望学习率大一点，方便快速前进；训练后期通常希望学习率小一点，方便更稳定地收敛。

（5）训练循环

训练循环的本质就是反复做：
* 看一批数据
* 算预测结果
* 算损失
* 算梯度
* 更新参数

不断重复之后，模型就会逐渐学会任务。

#### 3. MLP 模型结构设计

##### 3.1 输入层为什么是 784
MNIST 的每张图片原始大小是：

`28 × 28`

MLP 不能直接处理二维图片，所以要先展平：

`28 * 28 = 784`

也就是说，输入层神经元个数应该和输入特征维度一致。


##### 3.2 输出层为什么是 10
因为我们的任务是识别数字：
* 0
* 1
* 2
* …
* 9

一共 10 个类别，所以输出层需要：

`10 个神经元`

每个输出位置对应一个类别的得分。

##### 3.3 隐藏层怎么设计
隐藏层没有唯一标准答案，它属于一种超参数设计。

对于 MNIST 这种入门案例，我们可以先设计一个简单的 MLP，例如：

`784 → 256 → 128 → 10`

含义是：
* 输入层：784
* 隐藏层1：256
* 隐藏层2：128
* 输出层：10

这是一个非常典型、容易理解的结构。

##### 3.4 激活函数用什么
隐藏层通常使用：

`ReLU`

原因是：
* 计算简单
* 收敛较快
* 是最常见的基础激活函数之一

##### 3.5 为什么可以加入 Batch Normalization
在隐藏层后面，我们经常会加入：

`BatchNorm1d`

因为当前输入到全连接层的是二维张量：

`[batch_size, feature_dim]`

这正适合使用一维批标准化：

`nn.BatchNorm1d(特征数)`

例如：
* 第一层输出 256 维，就可以写 nn.BatchNorm1d(256)
* 第二层输出 128 维，就可以写 nn.BatchNorm1d(128)

常见顺序是：

`Linear → BatchNorm → ReLU`

也就是：
1. 先线性变换
2. 再做批标准化
3. 再经过激活函数

#### 4. 使用 PyTorch 搭建MLP 模型

In [ ]:
import torch.nn as nn

class model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256) # 输入层到隐藏层
        self.bn1 = nn.BatchNorm1d(256) # 批归一化层
        self.ac1 = nn.ReLU() # 激活函数

        self.fc2 = nn.Linear(256, 128) # 隐藏层到隐藏层
        self.bn2 = nn.BatchNorm1d(128) # 批归一化层
        self.ac2 = nn.ReLU() # 激活函数

        self.fc3 = nn.Linear(128, 10) # 隐藏层到输出层

    def forward(self, x):
        x = x.view(-1, 28*28) # 将输入展平为一维向量
        x = self.fc1(x) # 输入层到隐藏层
        x = self.bn1(x) # 批归一化
        x = self.ac1(x) # 激活函数

        x = self.fc2(x) # 隐藏层到隐藏层
        x = self.bn2(x) # 批归一化
        x = self.ac2(x) # 激活函数

        x = self.fc3(x) # 隐藏层到输出层
        return x

#### 5. 实例化模型、损失函数、优化器与 scheduler

In [3]:
# 1. 实例化模型

model = model()

# 2. 定义损失函数和优化器
criterion = nn.CrossEntropyLoss() # 交叉熵损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Adam优化器

# 3. 定义scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer, # 优化器
    mode='min', # 监控指标的模式，'min'表示当监控指标停止下降时调整学习率
    factor=0.1, # 学习率调整的乘数
    patience=5, # 监控指标停止改善的耐心次数
    verbose=True # 是否打印学习率调整信息
)

/home/zhang/miniconda3/envs/da/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


#### 6. 为什么这里不用手动声明 Softmax

##### 6.1 因为我们使用的是 nn.CrossEntropyLoss()
在这个案例中，我们通常这样定义损失函数：

`criterion = nn.CrossEntropyLoss()`

这个损失函数在 PyTorch 中非常特殊，它已经把 Softmax 的相关计算整合进去了。

更准确地说，它内部相当于做了：

`log_softmax + NLLLoss`

所以：

👉 模型输出层只需要给出原始得分 logits，不需要你自己再手动做 Softmax。

##### 6.2 如果手动写了 Softmax，反而可能有问题
如果你在模型最后手动写：

`x = torch.softmax(x, dim=1)`

然后又把结果传给：

`nn.CrossEntropyLoss()`

就相当于重复处理了一次概率变换。

#### 7. 为什么这里不用手动把标签转成 one-hot

##### 7.1 因为 CrossEntropyLoss 期待的标签不是 one-hot
对于多分类任务，PyTorch 的 nn.CrossEntropyLoss() 要求：
* 模型输出：[batch_size, num_classes]
* 标签输入：[batch_size]，其中每个元素是类别编号

例如：

`labels = tensor([3, 7, 1, 0])`

这表示这 4 个样本的真实类别分别是：
* 第 1 个样本是 3
* 第 2 个样本是 7
* 第 3 个样本是 1
* 第 4 个样本是 0

#### 8. 定义训练方法和测试方法

In [7]:
def train(model, train_loader, criterion, optimizer):
    model.train() # 设置模型为训练模式
    total_loss = 0.0 # 初始化总损失
    for X_train, y_train in train_loader:
        optimizer.zero_grad() # 清零梯度
        y_pred = model(X_train) # 前向传播
        loss = criterion(y_pred, y_train) # 计算损失
        loss.backward() # 反向传播
        optimizer.step() # 更新参数
        total_loss += loss.item() # 累加损失
    average_loss = total_loss / len(train_loader) # 计算平均损失
    return average_loss # 返回平均损失

In [5]:
def val(model, test_loader, criterion):
    model.eval() # 设置模型为评估模式
    total_loss = 0.0 # 初始化总损失
    with torch.no_grad(): # 禁用梯度计算
        for X_test, y_test in test_loader:
            y_pred = model(X_test) # 前向传播
            loss = criterion(y_pred, y_test) # 计算损失
            total_loss += loss.item() # 累加损失
    average_loss = total_loss / len(test_loader) # 计算平均损失
    return average_loss # 返回平均损失

#### 9. 训练主循环

In [8]:
epochs = 30 # 定义训练轮数
for epoch in range(epochs):
    train_loss = train(model, train_loader, criterion, optimizer) # 训练模型并获取训练损失
    val_loss = val(model, test_loader, criterion) # 验证模型并获取验证损失
    scheduler.step(val_loss) # 更新学习率
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}') # 打印训练和验证损失

Epoch 1/30, Train Loss: 0.2052, Val Loss: 0.0983
Epoch 2/30, Train Loss: 0.0845, Val Loss: 0.0729
Epoch 3/30, Train Loss: 0.0578, Val Loss: 0.0781
Epoch 4/30, Train Loss: 0.0439, Val Loss: 0.0680
Epoch 5/30, Train Loss: 0.0361, Val Loss: 0.0582
Epoch 6/30, Train Loss: 0.0306, Val Loss: 0.0689
Epoch 7/30, Train Loss: 0.0266, Val Loss: 0.0580
Epoch 8/30, Train Loss: 0.0224, Val Loss: 0.0639
Epoch 9/30, Train Loss: 0.0198, Val Loss: 0.0655
Epoch 10/30, Train Loss: 0.0170, Val Loss: 0.0654
Epoch 11/30, Train Loss: 0.0164, Val Loss: 0.0613
Epoch 12/30, Train Loss: 0.0123, Val Loss: 0.0706
Epoch 13/30, Train Loss: 0.0148, Val Loss: 0.0662
Epoch 14/30, Train Loss: 0.0071, Val Loss: 0.0574
Epoch 15/30, Train Loss: 0.0043, Val Loss: 0.0535
Epoch 16/30, Train Loss: 0.0030, Val Loss: 0.0535
Epoch 17/30, Train Loss: 0.0022, Val Loss: 0.0556
Epoch 18/30, Train Loss: 0.0024, Val Loss: 0.0536
Epoch 19/30, Train Loss: 0.0018, Val Loss: 0.0560
Epoch 20/30, Train Loss: 0.0017, Val Loss: 0.0530
Epoch 21/

#### 10. 这几步的关系一定要真正理解

##### 10.1 训练中最核心的链条
```
前向传播
   ↓
计算损失
   ↓
梯度清零
   ↓
反向传播
   ↓
参数更新
   ↓
学习率调整（按策略）
```

##### 10.2 对应到代码中
``` python
outputs = model(images)
loss = criterion(outputs, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()
scheduler.step()   # 通常在 epoch 末尾调用
```